# 04 — ML Driver Model

This notebook walks through the machine learning component of the
quantamental engine:

1. **Feature/target matrix** — merging financial metrics + NLP features
2. **No-lookahead validation** — verifying point-in-time constraints
3. **Model training** — Ridge or ElasticNet regression
4. **Walk-forward validation** — expanding-window out-of-sample testing
5. **Baseline comparison** — 4 naive strategies for honest benchmarking
6. **Model audit** — honest disclosure of limitations

Module: `src/ml_models.py`

> **Key principle:** The ML model is interpretable and honestly evaluated.
> If it doesn't beat baselines, we say so. The value lies in feature
> organisation, scenario discipline, and auditability — not raw prediction.

> **Note:** The authoritative reproducibility path is `python scripts/run_pipeline.py --ticker NVDA --report-date 2026-05-01 --price-date 2026-05-01 --output-format both`. This notebook is an explanatory wrapper that inspects the same pipeline outputs.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from pathlib import Path

from src.config import get_default_config
from src.ml_models import MLDriverModel

In [ ]:
config = get_default_config()
ml = MLDriverModel(config)

print(f"Target:         {config.ml_target}")
print(f"Primary model:  {config.primary_model}")
print(f"Min train years: {config.min_train_years}")
print(f"Baselines:      {config.baselines}")

## 1. Feature/Target Matrix

`build_feature_target_matrix()` merges financial metrics and NLP features
into a single wide-format table. The target (next-period revenue growth)
is shifted forward so each row predicts the *next* period.

Critical columns:
- `feature_available_date` — when the features became public
- `prediction_date` — the date we "make" the prediction
- `target_available_date` — when the target value became public

A valid row requires: `feature_available_date ≤ prediction_date` AND
`target_available_date > prediction_date`.

In [ ]:
matrix_path = config.processed_dir / "ml_feature_target_matrix.csv"

if matrix_path.exists():
    matrix = pd.read_csv(matrix_path)
    print(f"Matrix shape: {matrix.shape}")
    print(f"Feature periods: {matrix['feature_period'].tolist()}")
    print(f"Target: {matrix['target_name'].iloc[0]}")
    matrix[["feature_period", "feature_available_date", "prediction_date",
            "target_period", "target_available_date", "target_value"]].head(10)
else:
    print("No feature/target matrix yet — run the pipeline first.")
    matrix = None

## 2. No-Lookahead Validation

`validate_no_lookahead_matrix()` checks every row for point-in-time
compliance. This is the most important integrity check in the pipeline.

In [ ]:
if matrix is not None:
    is_valid = ml.validate_no_lookahead_matrix(matrix)
    print(f"No-lookahead validation: {'PASSED ✅' if is_valid else 'FAILED ❌'}")

## 3. Model Training

We train a Ridge or ElasticNet model on the feature matrix.
The model is intentionally simple and interpretable — coefficients
directly show which features drive the prediction.

In [ ]:
if matrix is not None:
    # Identify feature columns (exclude metadata)
    meta_cols = {
        "feature_period", "feature_available_date", "prediction_date",
        "target_period", "target_available_date", "target_name",
        "target_value", "source_accessions",
    }
    feature_cols = [c for c in matrix.columns if c not in meta_cols]
    
    X = matrix[feature_cols].apply(pd.to_numeric, errors="coerce")
    y = pd.to_numeric(matrix["target_value"], errors="coerce")
    
    model, coefficients = ml.train_primary_model(X, y)
    
    if model is not None:
        print(f"Model type: {type(model).__name__}")
        print(f"Features: {len(feature_cols)}")
        print(f"\nTop 10 coefficients (by magnitude):")
        sorted_coefs = sorted(coefficients.items(), key=lambda x: abs(x[1]), reverse=True)
        for feat, coef in sorted_coefs[:10]:
            print(f"  {feat:40s} {coef:+.6f}")
    else:
        print("Insufficient data to train model.")

## 4. Walk-Forward Validation

Expanding-window walk-forward validation: train on all prior data,
predict the next period. Minimum training window = 3 periods.
This is the honest out-of-sample test.

In [ ]:
if matrix is not None and model is not None:
    wf_results = ml.walk_forward_validate(X, y)
    
    if wf_results["folds"]:
        print(f"Walk-forward folds: {len(wf_results['folds'])}")
        wf_eval = ml.evaluate(wf_results["predictions"], wf_results["actuals"])
        print(f"  MAE:                  {wf_eval['mae']}")
        print(f"  RMSE:                 {wf_eval['rmse']}")
        print(f"  Directional accuracy: {wf_eval['directional_accuracy']}")
    else:
        print("Not enough data for walk-forward validation.")

## 5. Baseline Comparison

Four naive baselines provide context for model performance:
- **last_period**: previous value
- **trailing_4q_avg**: average of last 4 values
- **three_year_avg**: average of last 3 values
- **linear_trend**: extrapolate a fitted line

In [ ]:
if matrix is not None:
    baselines = ml.compute_baselines(y)
    
    print("Baseline performance:")
    print(f"{'Baseline':25s} {'MAE':>10s} {'RMSE':>10s} {'Dir.Acc':>10s}")
    print("-" * 57)
    for bl_name, bl_preds in baselines.items():
        bl_eval = ml.evaluate(bl_preds, y.tolist())
        mae = f"{bl_eval['mae']:.6f}" if not np.isnan(bl_eval['mae']) else 'N/A'
        rmse = f"{bl_eval['rmse']:.6f}" if not np.isnan(bl_eval['rmse']) else 'N/A'
        da = f"{bl_eval['directional_accuracy']:.4f}" if not np.isnan(bl_eval.get('directional_accuracy', np.nan)) else 'N/A'
        print(f"  {bl_name:23s} {mae:>10s} {rmse:>10s} {da:>10s}")

## 6. Model Audit

`generate_model_audit()` writes an honest assessment to
`outputs/model_audit.md`. It includes:
- Model coefficients
- Walk-forward results vs baselines
- Honest disclosure if the model doesn't outperform baselines
- Annual-only caveat if applicable

In [ ]:
if matrix is not None and model is not None:
    audit_text = ml.generate_model_audit(
        model=model,
        baselines=baselines,
        features=X,
        walk_forward_results=wf_results if 'wf_results' in dir() else None,
        target=y,
    )
    # Show first 30 lines
    for line in audit_text.split("\n")[:30]:
        print(line)

---

**Next:** [05_valuation_and_report.ipynb](05_valuation_and_report.ipynb) —
DCF valuation, reverse-DCF grid, recommendation, and report generation.